In [ ]:
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from pymatgen.core import Composition, Element

import os
from dotenv import load_dotenv

load_dotenv()  
API_KEY = os.getenv("MP_API_KEY")
R_OXYGEN = 1.40  # Эффективный ионный радиус O2- по Шеннону (Å)

df = pd.read_csv("../data/oxides_features_magpie.csv")
print(f"Загружено нетоксичных оксидов: {len(df)}")

# ------------------------------------------------------------------
# 1. Ионные радиусы и геометрический толеранс-фактор (из формулы)
# ------------------------------------------------------------------
def get_cation_radius(el: Element) -> float:
    """Возвращает средний ионный радиус катиона (с откатом к атомному)."""
    r = el.average_ionic_radius
    if r is not None and float(r) > 0:
        return float(r)
    if el.atomic_radius is not None:
        return float(el.atomic_radius)
    if el.atomic_radius_calculated is not None:
        return float(el.atomic_radius_calculated)
    return np.nan

ionic_records = []
for formula in df["reduced_formula"]:
    comp = Composition(formula)
    cations = [(el, amt) for el, amt in comp.items() if el.symbol != "O"]
    
    radii, weights = [], []
    for el, amt in cations:
        r = get_cation_radius(el)
        if not np.isnan(r):
            radii.append(r)
            weights.append(amt)
            
    if len(radii) == 0:
        ionic_records.append({
            "r_cation_mean": np.nan, "r_cation_std": np.nan,
            "r_cation_min": np.nan, "r_cation_max": np.nan,
            "r_cation_O_ratio": np.nan, "octahedral_factor": np.nan,
            "tolerance_factor": np.nan, "n_cation_types": 0
        })
        continue

    radii = np.array(radii, dtype=float)
    weights = np.array(weights, dtype=float)
    w_norm = weights / weights.sum()
    
    r_mean = float(np.sum(radii * w_norm))
    r_std = float(np.sqrt(np.sum(w_norm * (radii - r_mean) ** 2)))
    r_min = float(np.min(radii))
    r_max = float(np.max(radii))
    
    # Обобщенный фактор Гольдшмидта: t = (r_A + r_O) / (sqrt(2) * (r_B + r_O)),
    # где r_A — радиус самого крупного катиона, r_B — самого мелкого
    t_factor = (r_max + R_OXYGEN) / (np.sqrt(2.0) * (r_min + R_OXYGEN))
    # Октаэдрический фактор: mu = r_B / r_O
    oct_factor = r_min / R_OXYGEN
    
    ionic_records.append({
        "r_cation_mean": r_mean,
        "r_cation_std": r_std,
        "r_cation_min": r_min,
        "r_cation_max": r_max,
        "r_cation_O_ratio": r_mean / R_OXYGEN,
        "octahedral_factor": oct_factor,
        "tolerance_factor": t_factor,
        "n_cation_types": len(cations)
    })

df_ionic = pd.DataFrame(ionic_records)
print("Ионные радиусы и толеранс-фактор рассчитаны.")

# ------------------------------------------------------------------
# 2. Догрузка 3D-структур из Materials Project и расчет координации
# ------------------------------------------------------------------
def extract_3d_features(structure) -> dict:
    """Считает координационные числа катионов по кислороду и длины связей."""
    if structure is None:
        return {}
    
    # Получаем всех соседей в радиусе 3.0 Å (быстро работает на Cython)
    all_nbrs = structure.get_all_neighbors(r=3.0)
    cns, mo_dists, mm_dists = [], [], []
    
    for site, nbrs in zip(structure.sites, all_nbrs):
        el_sym = site.species.elements[0].symbol
        if el_sym == "O":
            continue  # Смотрим окружение вокруг катионов
        
        o_dists = [n.nn_distance for n in nbrs if n.species.elements[0].symbol == "O"]
        m_dists = [n.nn_distance for n in nbrs if n.species.elements[0].symbol != "O"]
        
        if o_dists:
            d_min = min(o_dists)
            # Адаптивный порог первой координационной сферы: до 1.25 * d_min (не выше 3.0 Å)
            first_shell = [d for d in o_dists if d <= min(3.0, d_min * 1.25)]
            cns.append(len(first_shell))
            mo_dists.extend(first_shell)
        if m_dists:
            mm_dists.append(min(m_dists))
            
    return {
        "cn_cation_O_mean": float(np.mean(cns)) if cns else np.nan,
        "cn_cation_O_std": float(np.std(cns)) if cns else np.nan,
        "cn_cation_O_min": float(np.min(cns)) if cns else np.nan,
        "cn_cation_O_max": float(np.max(cns)) if cns else np.nan,
        "dist_MO_mean": float(np.mean(mo_dists)) if mo_dists else np.nan,
        "dist_MO_min": float(np.min(mo_dists)) if mo_dists else np.nan,
        "dist_MM_min": float(np.min(mm_dists)) if mm_dists else np.nan,
    }

mat_ids = df["material_id"].tolist()
chunk_size = 500
struct_records = []

print("Подгрузка 3D-структур и расчет координационных чисел...")
with MPRester(API_KEY, mute_progress_bars=True) as mpr:
    for start in range(0, len(mat_ids), chunk_size):
        chunk = mat_ids[start:start + chunk_size]
        docs = mpr.materials.summary.search(
            material_ids=chunk,
            fields=["material_id", "structure", "density", "volume", "nsites"]
        )
        for d in docs:
            feats = extract_3d_features(d.structure)
            feats["material_id"] = str(d.material_id)
            feats["density"] = d.density
            feats["volume_per_atom"] = (d.volume / d.nsites) if (d.volume and d.nsites) else np.nan
            struct_records.append(feats)
        if (start // chunk_size + 1) % 5 == 0 or start + chunk_size >= len(mat_ids):
            print(f"Обработано: {min(start + chunk_size, len(mat_ids))} / {len(mat_ids)}")

df_struct = pd.DataFrame(struct_records)

# Объединяем все признаки и сохраняем
df_full = pd.concat([df.reset_index(drop=True), df_ionic], axis=1)
df_full = df_full.merge(df_struct, on="material_id", how="left")
df_full.to_csv("../data/oxides_features_full.csv", index=False)
print(f"Сохранен файл oxides_features_full.csv размером {df_full.shape}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score

df = pd.read_csv("../data/oxides_features_magpie.csv")

# 1. Формируем группы признаков (убираем константные столбцы Magpie)
magpie_cols = [c for c in df.columns if c.startswith("MagpieData") and df[c].nunique() > 1]

struct_no_sym_cols = [
    "r_cation_mean", "r_cation_std", "r_cation_min", "r_cation_max",
    "r_cation_O_ratio", "octahedral_factor", "tolerance_factor", "n_cation_types",
    "cn_cation_O_mean", "cn_cation_O_std", "cn_cation_O_min", "cn_cation_O_max",
    "dist_MO_mean", "dist_MO_min", "dist_MM_min",
    "density", "volume_per_atom"
]

cs_dummies = pd.get_dummies(df["crystal_system"], prefix="cs", dtype=float)
X_sym = pd.concat([df[["spacegroup_number"]], cs_dummies], axis=1)

feature_sets = {
    "1. Только Magpie (состав)": df[magpie_cols],
    "2. Magpie + структура БЕЗ симметрии": df[magpie_cols + struct_no_sym_cols],
    "3. Magpie + структура + симметрия": pd.concat([df[magpie_cols + struct_no_sym_cols], X_sym], axis=1),
    "4. Только симметрия (контроль)": X_sym,
}

# Выбираем целевую переменную (DFT-метки из Materials Project)
# При необходимости можно поменять на "target_any_magnetic"
TARGET = "target_ferro"
y = df[TARGET]
groups = df["chemsys"]
gkf = GroupKFold(n_splits=5)

def build_models():
    return {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42, n_jobs=-1))
        ])
    }

# 2. Сравнение наборов признаков и моделей (5-fold GroupKFold по chemsys)
print(f"=== Сравнение моделей и тест на утечку через симметрию (Target: {TARGET}) ===\n")
results = []
oof_preds = {}
rf_importances = {}

for fs_name, X_curr in feature_sets.items():
    for model_name, model in build_models().items():
        aucs, f1s = [], []
        oof = np.zeros(len(df))
        imp = np.zeros(X_curr.shape[1])
        
        for tr, te in gkf.split(X_curr, y, groups=groups):
            model.fit(X_curr.iloc[tr], y.iloc[tr])
            proba = model.predict_proba(X_curr.iloc[te])[:, 1]
            preds = (proba >= 0.5).astype(int)
            oof[te] = proba
            aucs.append(roc_auc_score(y.iloc[te], proba))
            f1s.append(f1_score(y.iloc[te], preds))
            if model_name == "Random Forest":
                imp += model.named_steps["clf"].feature_importances_ / 5.0
                
        results.append({
            "Набор признаков": fs_name,
            "Модель": model_name,
            "ROC-AUC": f"{np.mean(aucs):.4f} ± {np.std(aucs):.4f}",
            "F1-score": f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}"
        })
        if model_name == "Random Forest":
            oof_preds[fs_name] = oof
            rf_importances[fs_name] = pd.Series(imp, index=X_curr.columns).sort_values(ascending=False)

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

# 3. Контроль качества ВНУТРИ групп с одинаковой симметрией (по сингониям)
print("\n=== Проверка внутри сингоний (Random Forest: БЕЗ симметрии vs С симметрией) ===")
oof_no_sym = oof_preds["2. Magpie + структура БЕЗ симметрии"]
oof_with_sym = oof_preds["3. Magpie + структура + симметрия"]

for cs, idx in df.groupby("crystal_system").groups.items():
    sub_y = y.loc[idx]
    auc_no = roc_auc_score(sub_y, oof_no_sym[idx])
    auc_yes = roc_auc_score(sub_y, oof_with_sym[idx])
    print(f"  {cs:14s} (n={len(idx):4d}) | Без SG: {auc_no:.4f} | С SG: {auc_yes:.4f} | Delta: {auc_yes - auc_no:+.4f}")

# 4. Топ-15 важнейших признаков полной модели
print("\n=== Топ-15 признаков (Random Forest: Magpie + структура + симметрия) ===")
print(rf_importances["3. Magpie + структура + симметрия"].head(15).round(4).to_string())